# Multimodal Keratoconus Detection — CornOrb

**Architecture:** ResNet-18 image branch + ClinicalMLP branch → Late fusion → Binary classification (Normal vs. Keratoconus)

All code lives in the repo modules. This notebook is a thin orchestration layer.

## 0. Setup

In [ ]:
# Clone repo and install the only missing dependency
!git clone https://github.com/your-username/CornOrb-Multimodal-KC.git
%cd CornOrb-Multimodal-KC
!pip install -q grad-cam


In [ ]:
import sys
sys.path.insert(0, ".")  # make repo root importable

from config import *
print(f"Device: {DEVICE}")
print(f"Clinical cols: {CLINICAL_COLS}")


## 1. Load Dataset

Set `DATA_SOURCE = 'zenodo'` to download from Zenodo, or `'drive'` to load from Google Drive.
Update `DRIVE_CSV` / `DRIVE_ZIP` in `config.py` if using Drive.

In [ ]:
from data.download import load_data

DATA_SOURCE = "zenodo"   # <── change to "drive" if using Google Drive
load_data(source=DATA_SOURCE)


## 2. Imports & EDA

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from scipy import stats
from statsmodels.stats.multitest import multipletests


In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
print("\nClass distribution:")
print(df["label"].value_counts())
df.head()


In [ ]:
# ── Class balance + sex distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

counts = df["label"].value_counts()
axes[0].bar(["Normal", "Keratoconus"], counts.values, color=["steelblue", "tomato"])
axes[0].set_title("Class Distribution"); axes[0].set_ylabel("Number of eyes")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

sex_counts = df["gender"].value_counts()
axes[1].bar(sex_counts.index.astype(str), sex_counts.values, color=["mediumpurple", "gold"])
axes[1].set_title("Sex Distribution"); axes[1].set_ylabel("Number of eyes")
for i, v in enumerate(sex_counts.values):
    axes[1].text(i, v + 5, str(v), ha="center", fontweight="bold")

sex_by_label = df.groupby(["label", "gender"]).size().unstack(fill_value=0)
sex_by_label.index = ["Normal", "Keratoconus"]
sex_by_label.plot(kind="bar", ax=axes[2], color=["mediumpurple", "gold"])
axes[2].set_title("Sex by Diagnosis"); axes[2].set_ylabel("Number of eyes")
axes[2].set_xlabel(""); axes[2].legend(title="Sex"); axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/eda_class_sex.png", dpi=150)
plt.show()


In [ ]:
# ── Box plots: clinical variables by diagnosis ────────────────────────────────
box_vars = {
    "age_years": "Age (years)", "kmax_value_D": "Kmax (D)",
    "pachy_central_um": "Central Pachymetry (µm)",
    "pachy_thinnest_um": "Thinnest Pachymetry (µm)",
    "astig_value_D": "Astigmatism (D)",
    "asphericity_anterior": "Anterior Asphericity",
    "asphericity_posterior": "Posterior Asphericity",
}
n_cols = 4
n_rows = int(np.ceil(len(box_vars) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4.5*n_rows))
axes = axes.flatten()
palette = {0: "#4C72B0", 1: "#C44E52"}
for idx, (col, label) in enumerate(box_vars.items()):
    sns.boxplot(data=df, x="label", y=col, hue="label",
                palette=palette, legend=False, ax=axes[idx], width=0.5)
    axes[idx].set_xticks([0,1]); axes[idx].set_xticklabels(["Normal","Keratoconus"])
    axes[idx].set_xlabel(""); axes[idx].set_title(label, fontsize=11)
for idx in range(len(box_vars), len(axes)):
    axes[idx].axis("off")
plt.suptitle("Clinical Variables by Diagnosis", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/clinical_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Table 2: Summary statistics with BH-adjusted p-values ────────────────────
table2_vars = {
    "age_years": "Age (years)", "pachy_central_um": "Central pachymetry (µm)",
    "pachy_thinnest_um": "Thinnest pachymetry (µm)", "kmax_value_D": "Kmax (D)",
    "astig_value_D": "Astigmatism (D)", "asphericity_anterior": "Anterior asphericity",
    "asphericity_posterior": "Posterior asphericity",
}
rows, pvals = [], []
for col, label in table2_vars.items():
    n_vals = df.loc[df["label"]==0, col].dropna()
    k_vals = df.loc[df["label"]==1, col].dropna()
    _, p = stats.ttest_ind(n_vals, k_vals, equal_var=False)
    pvals.append(p)
    rows.append({"Variable": label,
                 f"Normal (n={len(n_vals)})": f"{n_vals.mean():.1f} ± {n_vals.std():.1f}",
                 f"Keratoconus (n={len(k_vals)})": f"{k_vals.mean():.1f} ± {k_vals.std():.1f}"})
_, pvals_adj, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")
for i, row in enumerate(rows):
    row["pBH"] = "< 0.001" if pvals_adj[i] < 0.001 else f"{pvals_adj[i]:.3f}"
table2_df = pd.DataFrame(rows)
table2_df.to_csv(f"{OUTPUT_DIR}/table2_clinical_summary.csv", index=False)
table2_df


## 3. Preprocessing & Splits

In [ ]:
from preprocessing.splits import make_splits, verify_no_leakage

train_df, val_df, test_df, scaler = make_splits(df, clinical_cols=CLINICAL_COLS)
verify_no_leakage(train_df, val_df, test_df)


## 4. DataLoaders

In [ ]:
from data.dataset import build_dataloaders

train_loader, val_loader, test_loader = build_dataloaders(
    train_df, val_df, test_df, CLINICAL_COLS, batch_size=BATCH_SIZE
)

imgs, clin, lbls = next(iter(train_loader))
print(f"Image batch : {imgs.shape}   ← (B, 12, 224, 224)")
print(f"Clinical    : {clin.shape}   ← (B, {len(CLINICAL_COLS)})")
print(f"Labels      : {lbls.shape}")


## 5. Model

In [ ]:
from models.fusion_net import MultimodalFusionNet

# Quick parameter count
m = MultimodalFusionNet(n_clinical_features=len(CLINICAL_COLS), mode="fused").to(DEVICE)
total = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"Trainable parameters: {total:,}")
del m


## 6. Ablation Study

Set `RUN_MODE = 'train'` to train from scratch, or `'eval_only'` to load saved weights.
Update `PRETRAINED_WEIGHTS` in `config.py` if using Drive paths.

In [ ]:
from training.ablation import run_ablation

RUN_MODE = "train"   # <── change to "eval_only" to load saved weights

results = run_ablation(
    train_loader, val_loader, test_loader,
    run_mode=RUN_MODE
)


## 7. Evaluation

In [ ]:
from evaluation.metrics import (
    print_classification_reports,
    plot_training_curves,
    plot_confusion_matrices,
    plot_roc_curves,
    summary_table,
)

plot_training_curves(results)
print_classification_reports(results)
plot_confusion_matrices(results)
plot_roc_curves(results)
summary_table(results)


## 8. Error Analysis

In [ ]:
from evaluation.error_analysis import export_misclassified, clinical_feature_comparison

misclassified_df = export_misclassified(test_df, results, mode="fused")
clinical_feature_comparison(test_df, results, mode="fused")


## 9. Grad-CAM

In [ ]:
from evaluation.gradcam import run_gradcam_analysis

fused_model = results["fused"]["model"]
run_gradcam_analysis(fused_model, results, test_df, clinical_cols=CLINICAL_COLS, n_top=5)
